LAVAGRID ENVIRONMENT SETUP

In [1]:
import gymnasium as gym
import matplotlib.pyplot as plt
from minigrid.wrappers import FullyObsWrapper, RGBImgPartialObsWrapper

In [36]:
env = FullyObsWrapper(gym.make("MiniGrid-LavaCrossingS11N5-v0"))
obs, _ = env.reset()
obs['image'].shape

(11, 11, 3)

In [37]:
env_obs = FullyObsWrapper(env)
obs, _ = env_obs.reset()
obs['image'].shape

(11, 11, 3)

In [ ]:
def policy(observation):
    return env.action_space.sample()
#env = gym.make("MiniGrid-MultiRoom-N6-v0", render_mode="human")
env = gym.make("MiniGrid-LavaGapS7-v0", render_mode="human")
env = FullyObsWrapper(env)
# env = RGBImgPartialObsWrapper(env)
observation, info = env.reset(seed=42)
print(observation['image'].shape)
for _ in range(1):
    action = policy(observation)  # User-defined policy function
    observation, reward, terminated, truncated, info = env.step(action)
    print(observation)
    if terminated or truncated:
        observation, info = env.reset()
env.close()

(7, 7, 3)
{'image': array([[[ 2,  5,  0],
        [ 2,  5,  0],
        [ 2,  5,  0],
        [ 2,  5,  0],
        [ 2,  5,  0],
        [ 2,  5,  0],
        [ 2,  5,  0]],

       [[ 2,  5,  0],
        [10,  0,  0],
        [ 1,  0,  0],
        [ 1,  0,  0],
        [ 1,  0,  0],
        [ 1,  0,  0],
        [ 2,  5,  0]],

       [[ 2,  5,  0],
        [ 9,  0,  0],
        [ 9,  0,  0],
        [ 9,  0,  0],
        [ 1,  0,  0],
        [ 9,  0,  0],
        [ 2,  5,  0]],

       [[ 2,  5,  0],
        [ 1,  0,  0],
        [ 1,  0,  0],
        [ 1,  0,  0],
        [ 1,  0,  0],
        [ 1,  0,  0],
        [ 2,  5,  0]],

       [[ 2,  5,  0],
        [ 1,  0,  0],
        [ 1,  0,  0],
        [ 1,  0,  0],
        [ 1,  0,  0],
        [ 1,  0,  0],
        [ 2,  5,  0]],

       [[ 2,  5,  0],
        [ 1,  0,  0],
        [ 1,  0,  0],
        [ 1,  0,  0],
        [ 1,  0,  0],
        [ 8,  1,  0],
        [ 2,  5,  0]],

       [[ 2,  5,  0],
        [ 2,  5,  0],


---

## Prioritized Replay Buffer

In [40]:
from collections import deque

class PrioritizedReplayBuffer:
    def __init__(
        self, capacity, alpha=0.6, beta=0.4, beta_increment=0.001, epsilon=0.01
    ):
        self.capacity = capacity
        self.memory = deque(maxlen=capacity)
        self.alpha, self.beta, self.beta_increment, self.epsilon = (alpha, beta, beta_increment, epsilon)
        self.priorities = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        experience_tuple = (state, action, reward, next_state, done)
        # Initialize the transition's priority
        max_priority = int(self.capacity) # MODIFY
        self.memory.append(experience_tuple)
        self.priorities.append(max_priority)
    
    def update_priorities(self, indices, td_errors):
        for idx, td_error in zip(indices, td_errors.tolist()):
            # Update the transition's priority
            self.priorities[idx] = idx # MODIFY

    def increase_beta(self):
        # Increase beta if less than 1
        self.beta = 1 # MODIFY

    def __len__(self):
        return len(self.memory)


Test the Replay Buffer

In [ ]:
buffer = PrioritizedReplayBuffer(capacity=3)
buffer.push(state=[1,3], action=2, reward=1, next_state=[2,4], done=False)


Transition in memory buffer: deque([([1, 3], 2, 1, [2, 4], False)], maxlen=3)
Priority buffer: deque([3], maxlen=3)


In [ ]:
print("Transition in memory buffer:", buffer.memory)
print("Priority buffer:", buffer.priorities)

---

## PPO

In [28]:
import gymnasium as gym
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np


class Policy(nn.Module):
    continuous = True # you can change this

    def __init__(self, device=torch.device('cpu')):
        super().__init__()
        self.device = device

        self.stack_size = 4
        self.frame_stack = []

        self.conv = nn.Sequential(
            nn.Conv2d(3 * self.stack_size, 32, 8, stride=4),
            nn.ReLU(),
            nn.Conv2d(32, 64, 4, stride=2),
            nn.ReLU(),
            nn.Conv2d(64, 64, 3, stride=1),
            nn.ReLU(),
            nn.Flatten()
        )

        with torch.no_grad():
            dummy = torch.zeros(1, 3 * self.stack_size, 96, 96)
            conv_out = self.conv(dummy).shape[1]

        self.fc = nn.Sequential(
            nn.Linear(conv_out, 512),
            nn.ReLU()
        )

        self.action_dim = 3
        self.mu_head = nn.Linear(512, self.action_dim)
        self.log_std = nn.Parameter(torch.zeros(self.action_dim))
        self.value_head = nn.Linear(512, 1)

        self.optimizer = torch.optim.Adam(self.parameters(), lr=3e-4)

        self.log_prob = None
        self.value_scalar = None
        self.last_state = None

        self.to(self.device)

    def _preprocess_single(self, obs):
        arr = np.asarray(obs, dtype=np.float32).transpose(2, 0, 1)
        t = torch.tensor(arr, device=self.device)
        if t.max() > 1.0:
            t = t / 255.0
        return t

    def reset_stack(self, obs):
        frame = self._preprocess_single(obs)
        self.frame_stack = [frame for _ in range(self.stack_size)]

    def get_stacked_obs(self, obs):
        frame = self._preprocess_single(obs)
        self.frame_stack.append(frame)
        if len(self.frame_stack) > self.stack_size:
            self.frame_stack.pop(0)

        stacked = torch.cat(self.frame_stack, dim=0)

        return stacked.unsqueeze(0)

    def forward(self, x):
        h = self.conv(x)
        h = self.fc(h)
        mu = self.mu_head(h)
        value = self.value_head(h).squeeze(-1)
        return mu, self.log_std, value

    def act(self, state):
        if len(self.frame_stack) == 0:
            self.reset_stack(state)

        x = self.get_stacked_obs(state)
        self.last_state = x 

        mu, log_std, value = self.forward(x)
        std = log_std.exp()

        eps = torch.randn_like(mu)
        raw = mu + eps * std

        steer = torch.tanh(raw[:, 0])
        gas   = torch.sigmoid(raw[:, 1])
        brake = torch.sigmoid(raw[:, 2])

        action = np.array(
            [steer.item(), gas.item(), brake.item()],
            dtype=np.float32
        )

        normal = torch.distributions.Normal(mu, std)
        logp = normal.log_prob(raw)
        
        logp[:, 0] -= torch.log(1.0 - steer**2 + 1e-6)
        logp[:, 1] -= torch.log(gas * (1 - gas) + 1e-6)
        logp[:, 2] -= torch.log(brake * (1 - brake) + 1e-6)

        self.log_prob = logp.sum(-1).item()
        self.value_scalar = value.item()

        return action

    def train(self):
        n_updates = 300
        steps_per_update = 4000
        gamma = 0.99
        lam = 0.95
        ppo_epochs = 3
        clip_eps = 0.2
        minibatch = 64
        value_coef = 0.5
        entropy_coef = 0.01

        env = gym.make("CarRacing-v2", continuous=True)

        def compute_gae(rews, vals, dones, last_v):
            adv = np.zeros_like(rews)
            gae = 0.0
            for t in reversed(range(len(rews))):
                mask = 1.0 - dones[t]
                next_v = last_v if t == len(rews) - 1 else vals[t + 1]
                delta = rews[t] + gamma * next_v * mask - vals[t]
                gae = delta + gamma * lam * mask * gae
                adv[t] = gae
            return adv, adv + vals

        for upd in range(n_updates):
            obs, _ = env.reset()
            self.reset_stack(obs)

            states, actions, old_logp, values, rewards, dones = [], [], [], [], [], []
            episode_rewards = []
            ep_reward = 0.0

            for _ in range(steps_per_update):
                action = self.act(obs)

                states.append(self.last_state.squeeze(0).cpu().numpy())
                actions.append(action)
                old_logp.append(self.log_prob)
                values.append(self.value_scalar)

                obs, r, terminated, truncated, _ = env.step(action)
                done = terminated or truncated

                rewards.append(r)
                dones.append(done)
                rew = r
                ep_reward += rew

                if done:
                    episode_rewards.append(ep_reward)
                    ep_reward = 0.0
                    obs, _ = env.reset()
                    self.reset_stack(obs)

            with torch.no_grad():
                last_v = self.forward(self.get_stacked_obs(obs))[2].item()

            states_t = torch.tensor(np.array(states), dtype=torch.float32, device=self.device)
            actions_t = torch.tensor(np.array(actions), dtype=torch.float32, device=self.device)
            old_logp_t = torch.tensor(old_logp, dtype=torch.float32, device=self.device)

            adv, ret = compute_gae(
                np.array(rewards),
                np.array(values),
                np.array(dones, dtype=np.float32),
                last_v
            )

            adv = (adv - adv.mean()) / (adv.std() + 1e-8)

            states_t = torch.stack([torch.tensor(s, device=self.device, dtype=torch.float32) for s in states])
            actions_t = torch.tensor(np.array(actions), dtype=torch.float32, device=self.device)
            old_logp_t = torch.tensor(old_logp, dtype=torch.float32, device=self.device)
            adv_t = torch.tensor(adv, dtype=torch.float32, device=self.device)
            ret_t = torch.tensor(ret, dtype=torch.float32, device=self.device)

            idxs = np.arange(len(states))
            surrogate_values = []

            for _ in range(ppo_epochs):
                np.random.shuffle(idxs)
                for start in range(0, len(states), minibatch):
                    mb = idxs[start:start + minibatch]

                    s = states_t[mb]
                    a = actions_t[mb]
                    old_lp = old_logp_t[mb]
                    A = adv_t[mb]
                    R = ret_t[mb]

                    mu, log_std, v_pred = self.forward(s)
                    std = log_std.exp()
                    
                    eps = 1e-6
                    a[:, 0] = a[:, 0].clamp(-1 + eps, 1 - eps)
                    a[:, 1:3] = a[:, 1:3].clamp(eps, 1 - eps)
                    steer_raw = 0.5 * torch.log((1 + a[:, 0:1]) / (1 - a[:, 0:1]))
                    gas_raw   = torch.log(a[:, 1:2] / (1 - a[:, 1:2]))
                    brake_raw = torch.log(a[:, 2:3] / (1 - a[:, 2:3]))
                    raw = torch.cat([steer_raw, gas_raw, brake_raw], dim=-1)

                    dist = torch.distributions.Normal(mu, std)
                    logp = dist.log_prob(raw)

                    s1 = torch.sigmoid(raw[:, 1])
                    s2 = torch.sigmoid(raw[:, 2])
                    logp[:, 0] -= torch.log(1 - torch.tanh(raw[:, 0])**2 + 1e-6)
                    logp[:, 1] -= torch.log(s1 * (1 - s1) + 1e-6)
                    logp[:, 2] -= torch.log(s2 * (1 - s2) + 1e-6)

                    logp = logp.sum(-1)
                    ratio = torch.exp(logp - old_lp)

                    surr1 = ratio * A
                    surr2 = torch.clamp(ratio, 1 - clip_eps, 1 + clip_eps) * A
                    policy_loss = -torch.min(surr1, surr2).mean()

                    value_loss = F.mse_loss(v_pred, R)
                    entropy = dist.entropy().sum(-1).mean()

                    loss = policy_loss + value_coef * value_loss - entropy_coef * entropy

                    self.optimizer.zero_grad()
                    loss.backward()
                    nn.utils.clip_grad_norm_(self.parameters(), 0.5)
                    self.optimizer.step()

                    surrogate_values.append(policy_loss.item())

            mean_reward = np.mean(episode_rewards) if episode_rewards else 0.0
            print(f"[PPO] Update {upd+1}/{n_updates} completed | Mean episode reward: {mean_reward:.2f}")

        print("Training completed.")
        
    def save(self):
        torch.save(self.state_dict(), "model.pt")
        
    def load(self):
        self.load_state_dict(torch.load("model.pt", map_location=self.device))
        
    def to(self, device):
        ret = super().to(device)
        ret.device = device
        return ret

## torch test

In [34]:
import torch
import numpy as np

x = x = torch.zeros(2, 1, 1)
print(x.shape)
x

torch.Size([2, 1, 1])


tensor([[[0.]],

        [[0.]]])

In [35]:
x = x.squeeze(-1)
print(x.shape)
x

torch.Size([2, 1])


tensor([[0.],
        [0.]])